# PHASE 1: DATA AND ARCHITECTURE SETUP

Publaynet dataset
This dataset is used to recognize the tables 

So what  I did here is:

first streamlined the dataset from hugging face then took 1000 samples and stored as python list as small_ds

MJSynth DataSet

This dataset will be used to identify the cropped text below is the preprocessing of the dataset is done

In [ ]:
#  The publaynet dataset is being prepared
from datasets import load_dataset  # datasets is a hugging face library 
import pprint
from datasets import Dataset
import matplotlib.pyplot as plt #This is used to build the actual white canvas for the image to put
import matplotlib.patches as patches 

# Streamlining the pyblaynet dataset from hugging face datasets library
ds = load_dataset(
    "jordanparker6/publaynet",
    split="train",
    streaming=True
)

val_ds = load_dataset(
    "jordanparker6/publaynet",
    split="validation",
    streaming=True
)



# inspecting few samples
for i, sample in enumerate(ds):
    if i == 1:
        break
    print("KEYS: ",sample.keys())
    # depth usually means the number of boxes I want to go inside
    pprint.pprint(sample["annotations"],depth = 2)
    

# Taking 5000 raw data from the dataset and storing in array
ds=ds.take(5000)
# converting the python list to a hugging face dataset
small_ds=list(ds)
# Visualizing the 4th image 
sample = small_ds[3] 
image = sample["image"]
annotations = sample["annotations"]

# fig = figure and ax = axes 
# fig is the window that holds everything if I want to change the color or save the image then I can use fig
# ax is the plot the graphing area containing the x-axis, and the y-axis, the gridlines and the data.
fig, ax = plt.subplots(figsize=(10,10))
# imshow() stands for image show method is desinged to render 2d pictures.
ax.imshow(image)
for annotation in annotations:
    x, y, w, h = annotation["bbox"]
    rect = patches.Rectangle(
        (x, y),
        w, 
        h,
        linewidth=2,
        # edgecolor defines the border of the rectangle
        edgecolor='red',
        # facecolor defines the inside of the rectangle
        facecolor='none' 
    )
    ax.add_patch(rect)

plt.show()


# stream only 1000 validation dataset
val_ds = val_ds.take(1000)
# convert the hugging face dataset into python list
val_ds = list(val_ds)



In [ ]:
# Preprocessing of the publaynet dataset and preparing the dataloaders
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as F
from torchvision.transforms import v2
from datasets import load_dataset
from torchvision import tv_tensors


# The hugging face dataSet are already parsed so making the class dataset 
class publayNetDataset(Dataset):
    def __init__(self, small_ds, transform):
        self.small_ds = small_ds
        self.transform = transform
    
    def __len__(self):
        return len(self.small_ds)

    # Bundle the python box and label tensors into python dictionary
    # Finally return the processed image tensor and the target dictionary 

     # Below function load a image using ovencv or PIL library
    def __getitem__(self, index):
        sample = self.small_ds[index]
        image = sample['image']
        annotations = sample['annotations']

        boxes = []
        labels = []
        
        # finds labels and bbox with that particular image
        for ann in annotations:
            bbox = ann['bbox']
            category = ann['category_id']

            x_min, y_min, w, h = bbox
            x_max = x_min + w
            y_max = y_min + h

            # stored each coordinate of boxes in python list
            if w>0 and h>0:
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(category)

        # Ensure the box has width and height greater than 0 
        # Convert the python list into tensors, boxes into float tensors and labels into integer tensors
        if len(boxes) > 0:
            boxes_tensor = torch.as_tensor(boxes, dtype = torch.float32)
        else:
            boxes_tensor = torch.empty((0, 4), dtype = torch.float32)
            
        label_tensor = torch.as_tensor(labels, dtype = torch.int64) 

        # wrapping the tensor so v2 knows those coordinates are bouding boxes So it can resize them as well
        W, H = image.size
        boxes_tensor = tv_tensors.BoundingBoxes(
            boxes_tensor,
            format="XYXY",
            canvas_size=(H,W)
        )
        
        # Wrapping the tensor objects in the dictionaries
        target = {
            "boxes": boxes_tensor,
            "labels": label_tensor
        }
        if self.transform:
            image, target = self.transform(image, target)
                
        return image, target


# initializing the transform pipeline
my_detection_transforms = v2.Compose([
    v2.ToImage(),  # Converts image into tensor image wrapper          
    v2.Resize(size=(800, 800)), # Resizes image and automatically scales bounding boxes
    v2.ToDtype(torch.float32, scale=True) # Normalizes to [0, 1] range
])


# This collate fun is used to merge the individual images into a batch of images.
def crnn_collate_fn(batch):
    images = []
    targets = []

    # Loop through the batch
    for image, target in batch:
        images.append(image)
        targets.append(target)

    # Stack the images into a single batch tensor (Only works if they are all resized to the same size!)
    images_tensor = torch.stack(images)
    # Leave targets exactly as they are: a list of dictionaries!
    return images_tensor, targets

# instantiating the  training class dataset
dataset = publayNetDataset(
    small_ds = small_ds, 
    transform = my_detection_transforms
)

# plug it into the training dataLoader 
my_dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=crnn_collate_fn
)

# Creating the validation dataset class
val_dataset = publayNetDataset(
    small_ds=val_ds,
    transform=my_detection_transforms
)

# Preparing the dataloader for the valdataset
val_dataloader=DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=crnn_collate_fn
)
print("done for valdataloader")


In [ ]:
# Testing the preprocessed dataset

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

# 1. Grab a single batch of data from your dataloader
# iter() turns the dataloader into an iterator, next() pulls the first batch
images, targets = next(iter(my_dataloader))

# 2. Print the architecture of the batch to verify collate_fn
print("--- BATCH STRUCTURE ---")
print(f"Number of images stacked: {len(images)}")
print(f"Stacked Images Tensor Shape: {images.shape}") # Expected: [32, Channel, 800, 800]
print(f"Number of target dictionaries: {len(targets)}") # Expected: 32
print(f"Target 0 Keys: {targets[0].keys()}")

# 3. Extract the very first image and its matching target dictionary
image_0 = images[0]
target_0 = targets[0]

# PyTorch images are formatted as [Channels, Height, Width] (e.g., [3, 800, 800])
# Matplotlib requires images to be [Height, Width, Channels] to plot them.
# .permute(1, 2, 0) reshuffles the dimensions for us
image_to_plot = image_0.permute(1, 2, 0).numpy()

# 4. Set up the matplotlib plot
fig, ax = plt.subplots(1, figsize=(12, 12))
ax.imshow(image_to_plot)

# 5. Extract the boxes and labels from the target dictionary
boxes = target_0["boxes"].numpy()
labels = target_0["labels"].numpy()

# 6. Loop through every box in the image and draw it
print(f"\nFound {len(boxes)} tables/objects in Image 0.")
for box, label in zip(boxes, labels):
    # Unpack the XYXY coordinates
    x_min, y_min, x_max, y_max = box
    
    # Matplotlib needs width and height to draw a rectangle
    width = x_max - x_min
    height = y_max - y_min
    
    # Create the red bounding box
    rect = patches.Rectangle(
        (x_min, y_min), width, height, 
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    
    # Add the label number right above the box
    ax.text(
        x_min, y_min - 5, f"Label: {label}", 
        color='red', fontsize=12, weight='bold', 
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
    )

plt.title("PubLayNet Dataloader Test: Resized Image & Bounding Boxes", fontsize=16)
plt.axis('off')
plt.show()

# Phase 2 : Building the table detector
 First of all I am going to build a object detection setup using model YOLOv8 and then format dataloader into the publaynet dataset into yolo format and then train the model to detect the 2 specific classes table and table_cell.

In [ ]:
!pip install ultralytics

In [ ]:
# Training the yolo model
import os
import shutil
import cv2
import yaml
import torch
import numpy as np


base_dir = '/kaggle/working/publaynet_yolo'
 
# Create the folder structure
for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

# Making the EXport function for the publaynet dataloader 
def export_dataloader_to_yolo(dataloader, split_name, base_dir, class_mapping):
    """
    Iterates through a dataloader and saves images and bounding boxes in YOLO format.
    """
    print(f"Starting export for {split_name} split...")
    
    for batch_idx, (images, targets) in enumerate(dataloader):
        # Handle cases where targets might be batched differently depending on collate_fn
        for i in range(len(images)):
            
            # --- A. Save the Image ---
            # Assume images are PyTorch tensors (C, H, W) normalized between 0 and 1
            img_tensor = images[i].permute(1, 2, 0).cpu().numpy()
            img_bgr = cv2.cvtColor((img_tensor * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            
            # Get image dimensions for normalization (if needed later)
            img_h, img_w = img_bgr.shape[0:2]
            
            file_prefix = f"batch_{batch_idx}_img_{i}"
            img_path = f"{base_dir}/images/{split_name}/{file_prefix}.jpg"
            cv2.imwrite(img_path, img_bgr)
            
            # --- B. Save the Labels ---
            label_path = f"{base_dir}/labels/{split_name}/{file_prefix}.txt"
            
            # --- B. Save the Labels ---
            label_path = f"{base_dir}/labels/{split_name}/{file_prefix}.txt"
            
            with open(label_path, 'w') as f:
                try:
                    boxes = targets[i]['boxes'].cpu().numpy()
                    labels = targets[i]['labels'].cpu().numpy()
                    
                    for box, label in zip(boxes, labels):
                        # Skip if the label isn't in our mapping
                        if int(label) not in class_mapping:
                            continue
                            
                        yolo_class_id = class_mapping[int(label)]
                        
                        # Unpack the [xmin, ymin, xmax, ymax] coordinates
                        x_min, y_min, x_max, y_max = box
                        
                        # Auto-detect absolute pixels vs. already normalized coords
                        if x_max > 1.0 or y_max > 1.0:
                            # Absolute pixels: convert to center and normalize by image dimensions
                            x_c = ((x_min + x_max) / 2.0) / img_w
                            y_c = ((y_min + y_max) / 2.0) / img_h
                            w = (x_max - x_min) / img_w
                            h = (y_max - y_min) / img_h
                        else:
                            # Already normalized: just convert to center, width, and height
                            x_c = (x_min + x_max) / 2.0
                            y_c = (y_min + y_max) / 2.0
                            w = x_max - x_min
                            h = y_max - y_min
                        
                        # Ensure values are strictly within [0, 1] bounds to prevent YOLO errors
                        x_c = np.clip(x_c, 0.0, 1.0)
                        y_c = np.clip(y_c, 0.0, 1.0)
                        w = np.clip(w, 0.0, 1.0)
                        h = np.clip(h, 0.0, 1.0)
                        
                        f.write(f"{yolo_class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
                        
                except KeyError:
                    print(f"Error extracting boxes in batch {batch_idx}. Check dictionary keys.")
                    break
                    
    print(f"Finished exporting {split_name} split.")


# 3. EXECUTE EXPORT
# Define your class mapping based on PubLayNet IDs
# Example: If PubLayNet table ID is 4, and cell/text is 1
class_mapping = {4: 0, 1: 1} # Maps dataset ID -> YOLO 0-indexed ID

# Call the function export dataloader
export_dataloader_to_yolo(my_dataloader, 'train', base_dir, class_mapping)
export_dataloader_to_yolo(val_dataloader, 'val', base_dir, class_mapping)

# ==========================================
# 4. GENERATE DATA.YAML
# ==========================================
yaml_path = f'{base_dir}/data.yaml'

# Safely clean up old yaml files/folders if they exist
if os.path.exists(yaml_path):
    if os.path.isdir(yaml_path):
        shutil.rmtree(yaml_path) 
    else:
        os.remove(yaml_path)

yaml_data = {
    'train': f"{base_dir}/images/train",
    'val': f"{base_dir}/images/val", # Will be empty if you didn't export val_dataloader
    'nc': 2, 
    'names': ['table', 'table_cell'] 
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"Setup complete! Ready to train. YAML saved at: {yaml_path}")

In [ ]:
from ultralytics import YOLO
import shutil

# Load a pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train it pointing to the yaml file we just generated
results = model.train(
    data='/kaggle/working/publaynet_yolo/data.yaml',  
    epochs=50,
    imgsz=640,
    batch=16,
    project='/kaggle/working/yolo_runs',              
)

# this line is used to copy the soruce file in a destination folder
shutil.copy(
    '/kaggle/working/yolo_runs/train/weights/best.pt',
    '/kaggle/working/best.pt'   # ← shows up in "Output" tab
)

In [ ]:
# Checking the IoU for the trained YOLO model

from ultralytics import YOLO

# 1. model load
model = YOLO("/kaggle/working/best.pt")

metrics = model.val(data="/kaggle/working/publaynet_yolo/data.yaml", split="val")

print(f"mAP50 (IoU threshold at 50%): {metrics.box.map50:.4f}")
print(f"mAP50-95 (Average IoU from 50% to 95%): {metrics.box.map:.4f}")
